# 4) EDA baseline (schema, nulls, categories)


In [ ]:
# Inputs: processed CSVs | Process: load satcat and gp as DataFrames | Outputs: satcat_df, gp_df

import pandas as pd
from pathlib import Path
from IPython.display import HTML

# --- Editable settings ---
# processed_dir: where the processed CSVs live
processed_dir = Path("../data/processed")

# satcat filename (change if using a different file)
satcat_filename = "satcat_latest.csv"

# gp pattern/filename: picks the latest matching 'gp_*.csv' if available, else falls back to 'gp_latest.csv'
gp_glob_pattern = "gp_*.csv"
gp_fallback_filename = "gp_latest.csv"
# -------------------------

# Resolve paths
satcat_path = processed_dir / satcat_filename
gp_candidates = sorted(processed_dir.glob(gp_glob_pattern))
gp_path = gp_candidates[-1] if gp_candidates else (processed_dir / gp_fallback_filename)
satnav_path = processed_dir / "satnav_latest.csv"


# Safety checks
if not satcat_path.exists():
    raise FileNotFoundError(f"satcat file not found: {satcat_path}")
if not gp_path.exists():
    raise FileNotFoundError(f"gp file not found. Tried latest of '{gp_glob_pattern}' and fallback '{gp_fallback_filename}'. Checked: {processed_dir}")

# Load
satcat_df = pd.read_csv(satcat_path, low_memory=False)
gp_df = pd.read_csv(gp_path, low_memory=False)
satnav_df = pd.read_csv(satnav_path, low_memory=False)

print("Loaded satcat:", satcat_path)
print("Loaded gp:", gp_path)
print("Loaded satnav:", satnav_path)
print("satcat_df shape:", satcat_df.shape)
print("gp_df shape:", gp_df.shape)
print("satnav_df shape:", satnav_df.shape)

# Show all columns (and full column text) when using head(), etc.
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

def show_freeze_header(df, height=400):
    html = df.to_html(index=False)
    return HTML(f"""
    <style>
    .scrollbox {{ max-height:{height}px; overflow-y:auto; }}
    .scrollbox thead th {{ position: sticky; top: 0; background: white; z-index: 2; }}
    </style>
    <div class="scrollbox">{html}</div>
    """)

# Example:
satnav_df.head(10)

In [ ]:
# Inputs: satcat_df | Process: schema + nulls + describe | Outputs: prints
satnav_df.info()
satnav_df.isnull().sum()
satnav_df.describe()


In [ ]:
# Inputs: satcat_df | Process: categorical preview | Outputs: unique values
print(satcat_df.get('COUNTRY', pd.Series()).unique())


In [ ]:
# Inputs: satnav_df | Process: high-level schema + memory | Outputs: info + memory usage
satnav_df.info()
print("\nApprox. memory MB:", round(satnav_df.memory_usage(deep=True).sum() / 1e6, 2))

In [ ]:
# Inputs: satnav_df | Process: missingness per column | Outputs: table sorted by % missing
miss = satnav_df.isna().mean().sort_values(ascending=False).to_frame("pct_missing")
miss.head(20)

In [ ]:
# Inputs: gp_df | Process: cardinality per column | Outputs: nunique sorted
card2 = gp_df.nunique(dropna=True).sort_values(ascending=False)
card2.head(20), card2.tail(20)


In [ ]:
# Inputs: satnav_df | Process: cardinality per column | Outputs: nunique sorted
card = satcat_df.nunique(dropna=True).sort_values(ascending=False).to_frame("nunique")
card.head(20), card.tail(20)

In [ ]:
# Inputs: satnav_df | Process: quick sample | Outputs: head and tail
display(satnav_df.head(3))
display(satnav_df.tail(3))

In [ ]:
# Inputs: satnav_df | Process: top categories | Outputs: top 10 per selected categorical cols
for col in ["object_type", "site", "rcs_size", "country"]:
    if col in satnav_df.columns:
        print(f"\n{col} (top 10):")
        print(satnav_df[col].value_counts(dropna=False).head(10))

In [ ]:
# Inputs: satnav_df | Process: numeric hygiene (zeros/negatives) | Outputs: small table
import numpy as np
num_cols = satnav_df.select_dtypes(include=[np.number]).columns
stats = []
for c in num_cols:
    s = satnav_df[c]
    stats.append({
        "col": c,
        "min": s.min(),
        "max": s.max(),
        "zeros": int((s == 0).sum()),
        "<0": int((s < 0).sum()),
        "missing": int(s.isna().sum())
    })
pd.DataFrame(stats).sort_values(by=["<0","zeros","missing"], ascending=False).head(20)

In [ ]:
# Inputs: satnav_df | Process: parse dates + ranges | Outputs: min/max/invalid counts
date_cols = [c for c in ["launch", "decay", "launch_date", "decay_date", "epoch"] if c in satnav_df.columns]
out = {}
for c in date_cols:
    dt = pd.to_datetime(satnav_df[c], errors="coerce", utc=True)
    out[c] = {
        "min": dt.min(),
        "max": dt.max(),
        "invalid": int(dt.isna().sum())
    }
pd.DataFrame(out)

In [ ]:
# Inputs: satnav_df | Process: cross-field sanity checks | Outputs: counts
from pandas import to_datetime
res = {}
if set(["launch_date","decay_date"]).issubset(satnav_df.columns):
    launch_dt = to_datetime(satnav_df["launch_date"], errors="coerce", utc=True)
    decay_dt  = to_datetime(satnav_df["decay_date"], errors="coerce", utc=True)
    res["decay_before_launch"] = int((decay_dt < launch_dt).sum())


In [ ]:
# Inputs: satnav_df | Process: IQR outlier scan on selected metrics | Outputs: outlier counts
check_cols = [c for c in ["period","inclination","apoapsis","periapsis","semimajor_axis","mean_motion"] if c in satnav_df.columns]
outliers = {}
for c in check_cols:
    s = satnav_df[c].dropna()
    if s.empty: 
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers[c] = int(((satnav_df[c] < lo) | (satnav_df[c] > hi)).sum())
outliers

In [ ]:
# Inputs: satnav_df | Process: correlations for numeric features | Outputs: correlation matrix (top-left)
num_cols = satnav_df.select_dtypes(include="number")
corr = num_cols.corr().round(3)
corr.iloc[:10, :10]

In [ ]:
# Inputs: satnav_df | Process: detect mixed-type strings in object cols | Outputs: potential mixed-type columns
suspect = []
for c in satnav_df.select_dtypes(include="object").columns:
    s = satnav_df[c].dropna().astype(str).str.strip()
    numeric_like = pd.to_numeric(s, errors="coerce").notna().mean() if len(s) else 0
    if 0.2 < numeric_like < 0.8:
        suspect.append((c, round(numeric_like, 2)))
print(suspect)

In [ ]:
# Inputs: satnav_df | Process: whitespace/non-printable anomalies | Outputs: counts per col
import re
def non_printable_count(series: pd.Series) -> int:
    return series.dropna().astype(str).apply(lambda x: int(bool(re.search(r"[^\\x20-\\x7E\\t\\n\\r]", x)))).sum()
rows = []
for c in satnav_df.select_dtypes(include="object").columns:
    s = satnav_df[c].astype(str)
    rows.append({
        "col": c,
        "leading_ws": int((s.str.match(r"\\s")).sum()),
        "trailing_ws": int((s.str.match(r".*\\s$")).sum()),
        "non_printable": non_printable_count(s)
    })
pd.DataFrame(rows).sort_values(by=["leading_ws","trailing_ws","non_printable"], ascending=False).head(20)

In [ ]:
# Inputs: satnav_df | Process: unique previews for key categorical columns | Outputs: unique lists (truncated)
def preview_unique(col, n=30):
    vals = satnav_df[col].dropna().astype(str).unique().tolist()
    print(f"{col} ({len(vals)} unique):", vals[:n])
for col in ["object_type","site","rcs_size","country","classification_type","ref_frame","time_system"]:
    if col in satnav_df.columns:
        preview_unique(col)

In [1]:
# Inputs: satcat_df | Process: build a data dictionary | Outputs: dataframe summary
summary = pd.DataFrame({
    "dtype": satcat_df.dtypes.astype(str),
    "nunique": satcat_df.nunique(dropna=True),
    "missing": satcat_df.isna().sum(),
    "pct_missing": (satcat_df.isna().mean()*100).round(2)
})
summary.sort_values(by=["pct_missing","nunique"], ascending=[False, False]).head(41)

NameError: name 'pd' is not defined

In [ ]:
# Purpose: Define functions
from IPython.display import HTML

from IPython.display import HTML

def show_freeze_header(df, height=400):
    html = df.to_html(index=False)
    return HTML(f"""
    <style>
    .scrollbox {{ max-height:{height}px; overflow-y:auto; }}
    .scrollbox thead th {{ position: sticky; top: 0; background: white; z-index: 2; }}
    </style>
    <div class="scrollbox">{html}</div>
    """)


show_freeze_header(gp_df.head(500).sort_values(by='NORAD_CAT_ID', ascending=True)   , height=400)

In [ ]:
# Purpose: Define functions
from IPython.display import HTML

from IPython.display import HTML

def show_freeze_header(df, height=400):
    html = df.to_html(index=False)
    return HTML(f"""
    <style>
    .scrollbox {{ max-height:{height}px; overflow-y:auto; }}
    .scrollbox thead th {{ position: sticky; top: 0; background: white; z-index: 2; }}
    </style>
    <div class="scrollbox">{html}</div>
    """)


show_freeze_header(satcat_df.head(500), height=400)

In [ ]:
# Purpose: Define functions
from IPython.display import HTML

from IPython.display import HTML

def show_freeze_header(df, height=400):
    html = df.to_html(index=False)
    return HTML(f"""
    <style>
    .scrollbox {{ max-height:{height}px; overflow-y:auto; }}
    .scrollbox thead th {{ position: sticky; top: 0; background: white; z-index: 2; }}
    </style>
    <div class="scrollbox">{html}</div>
    """)


show_freeze_header(gp_df.head(500), height=400)

In [ ]:
# Purpose: Define functions
from IPython.display import HTML

from IPython.display import HTML

def show_freeze_header(df, height=400):
    html = df.to_html(index=False)
    return HTML(f"""
    <style>
    .scrollbox {{ max-height:{height}px; overflow-y:auto; }}
    .scrollbox thead th {{ position: sticky; top: 0; background: white; z-index: 2; }}
    </style>
    <div class="scrollbox">{html}</div>
    """)


show_freeze_header(satnav_df.head(500), height=400)

In [ ]:
# Inputs: gp_df, list of columns | Process: drop columns, view | Outputs: satnav_clean HTML table
cols_to_remove = ['ELEMENT_SET_NO', 'CLASSIFICATION_TYPE', 'EPHEMERIS_TYPE', 'MEAN_ELEMENT_THEORY','TIME_SYSTEM', 'REF_FRAME', 'CENTER_NAME', 'ORIGINATOR', 'COMMENT', 'CCSDS_OMM_VERS']  # edit this list

gp_clean = gp_df.drop(columns=cols_to_remove, errors='ignore').copy()
print(gp_clean.shape)
show_freeze_header(
    gp_clean.head(500),
    height=400
)

In [ ]:
# Inputs: satcat_df, list of columns | Process: drop columns, view | Outputs: satnav_clean HTML table
cols_to_remove = ['CURRENT', 'RCSVALUE', 'COMMENTCODE']  # edit this list

satcat_clean = satcat_df.drop(columns=cols_to_remove, errors='ignore').copy()
print(satcat_clean.shape)

show_freeze_header(
    satcat_clean.head(500),
    height=400
)

In [ ]:
satnav_tidy = satnav_clean.assign(object_type=satnav_clean['OBJECT_TYPE_y'].combine_first(satnav_clean['OBJECT_TYPE_x'])).drop(columns=['OBJECT_TYPE_x','OBJECT_TYPE_y'])
satnav_tidy = satnav_clean.assign(site=satnav_clean['SITE_x'].combine_first(satnav_clean['SITE_y'])).drop(columns=['RCS_SIZE_y','RCS_SIZE_x'])
satnav_tidy = satnav_clean.assign(period=satnav_clean['PERIOD_y'].combine_first(satnav_clean['PERIOD_x'])).drop(columns=['PERIOD_y','PERIOD_x'])
satnav_tidy = satnav_clean.assign(inclination=satnav_clean['INCLINATION_y'].combine_first(satnav_clean['INCLINATION_x'])).drop(columns=['INCLINATION_y','INCLINATION_x'])
satnav_tidy = satnav_clean.assign(apoapsis=satnav_clean['APOAPSIS'].combine_first(satnav_clean['APOGEE'])).drop(columns=['APOAPSIS','APOGEE'])
satnav_tidy = satnav_clean.assign(periapsis=satnav_clean['PERIAPSIS'].combine_first(satnav_clean['PERIGEE'])).drop(columns=['PERIAPSIS','PERIGEE'])
satnav_tidy = satnav_clean.assign(rcs_size=satnav_clean['RCS_SIZE_y'].combine_first(satnav_clean['RCS_SIZE_x'])).drop(columns=['RCS_SIZE_y','RCS_SIZE_x'])
satnav_tidy = satnav_clean.assign(object_name=satnav_clean['OBJECT_NAME_x'].combine_first(satnav_clean['OBJECT_NAME_y'])).drop(columns=['OBJECT_NAME_y','OBJECT_NAME_x'])
satnav_tidy = satnav_clean.assign(object_id=satnav_clean['OBJECT_ID_x'].combine_first(satnav_clean['OBJECT_ID_y'])).drop(columns=['OBJECT_ID_x','OBJECT_ID_y'])

print(satnav_tidy.shape)

show_freeze_header(
    satnav_tidy.head(500),
    height=400
)

In [ ]:
# Inputs: satnav_tidy | Process: build a data dictionary | Outputs: dataframe summary
summary = pd.DataFrame({
    "dtype": satnav_tidy.dtypes.astype(str),
    "nunique": satnav_tidy.nunique(dropna=True),
    "missing": satnav_tidy.isna().sum(),
    "pct_missing": (satnav_tidy.isna().mean()*100).round(2)
})
print(summary.sort_values(by=["pct_missing","nunique"], ascending=[False, False]).head(41))

In [ ]:
# Purpose: Define functions
from IPython.display import HTML

from IPython.display import HTML

def show_freeze_header(df, height=400):
    html = df.to_html(index=False)
    return HTML(f"""
    <style>
    .scrollbox {{ max-height:{height}px; overflow-y:auto; }}
    .scrollbox thead th {{ position: sticky; top: 0; background: white; z-index: 2; }}
    </style>
    <div class="scrollbox">{html}</div>
    """)
col=['INTLDES', 'SATNAME']
anti_satcat = satnav_tidy[col]
show_freeze_header(anti_satcat.head(500).sort_values(by=col, ascending=True)   , height=400)
# Inputs: satnav_df | Process: detect duplicates by INTLDES key (here SATNAME used as proxy) | Outputs: counts + sample

In [ ]:
# Inputs: satnav_tidy | Process: save CSV | Outputs: file path
from pathlib import Path

out_path = Path("../data/processed/satnav_tidy.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
satnav_tidy.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())

In [ ]:
satnav_tidy.shape

In [ ]:
# Inputs: satnav_tidy, list of columns | Process: drop columns, view | Outputs: satnav_clean HTML table
cols_to_remove = ['CURRENT', 'RCSVALUE', 'COMMENTCODE']  # edit this list

satnav_tidy = satnav_tidy.drop(columns=cols_to_remove, errors='ignore').copy()
print(satcat_clean.shape)

show_freeze_header(
    satcat_clean.head(500),
    height=400
)

In [ ]:
# Inputs: satnav_clean | Process: combine _x/_y columns into single columns | Outputs: satnav_tidy
satnav_tidy = satnav_clean.copy()

# Combine _x/_y columns (prefer _y, fallback to _x)
combine_cols = [
    ('OBJECT_TYPE_y', 'OBJECT_TYPE_x', 'object_type'),
    ('SITE_y', 'SITE_x', 'site'),
    ('PERIOD_y', 'PERIOD_x', 'period'),
    ('INCLINATION_y', 'INCLINATION_x', 'inclination'),
    ('APOAPSIS', 'APOGEE', 'apoapsis'),
    ('PERIAPSIS', 'PERIGEE', 'periapsis'),
    ('RCS_SIZE_y', 'RCS_SIZE_x', 'rcs_size'),
    ('OBJECT_NAME_y', 'OBJECT_NAME_x', 'object_name'),
    ('OBJECT_ID_y', 'OBJECT_ID_x', 'object_id')
]

for col_y, col_x, new_col in combine_cols:
    satnav_tidy[new_col] = satnav_tidy[col_y].combine_first(satnav_tidy[col_x])
    satnav_tidy = satnav_tidy.drop(columns=[col_y, col_x], errors='ignore')

In [ ]:
# Inputs: satnav_tidy, list of columns | Process: drop specified columns | Outputs: updated satnav_tidy
cols_to_drop = ['GP_ID', 'TLE_LINE0', 'FILE_y', 'FILE_x']  # edit this list

satnav_tidy = satnav_tidy.drop(columns=cols_to_drop, errors='ignore')
print(f"Dropped {len(cols_to_drop)} columns")